# **VisDrone Human Detection Project**
## Project overview
This project uses the **VisDrone-DET2019** dataset in Google Colab Pro to detect people in drone imagery.
**Validation metrics (548 validation images):** Precision 0.7642, Recall 0.6207, mAP@50 0.6910, and mAP@50–95 0.3278. These values come from a separate evaluation of `best.pt`; they are not test-set results.
## Setup
**Environment:** Colab Pro  
**Runtime:** Python 3  
**Hardware:** A100 GPU  
**Model:** YOLO11m-P2  
**Training settings:** image size 832 | 150 target epochs | batch size 24

---
### Before you start
1. In Colab, choose **Runtime > Change runtime type > Python 3**.
2. Choose **A100 GPU** as the hardware accelerator.
---
### Getting started
1. Run all cells or execute them in order: 1 → 2 → 3 → 4.
2. Grant Google Drive access when prompted so files persist after the runtime ends.

| Section | Task |
|---|---|
| 1 | GPU check and setup |
| 2 | Dataset preparation |
| 3 | Training |
| 4 | Validation |
| 5 | Optional video preprocessing |
| 6 | ByteTrack inference |

---
# **SECTION 1 — GPU Check and Setup**
---

A GPU is required. Running this training workflow on a CPU may cause performance and resource problems.

In [ ]:
import torch

# Check for a GPU
if not torch.cuda.is_available():
    raise EnvironmentError(
        "GPU not found! Select a GPU in Colab runtime settings."
    )
gpu_name  = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

##### Install the required libraries. If the runtime disconnects and a package is missing later, rerun this setup cell.

In [ ]:
# Install required packages
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics', 'lapx', 'opencv-python', 'matplotlib'],
    check=True
)
print("Setup complete")


##### Connect Google Drive so outputs are preserved after the Colab session ends. Grant access when prompted.

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Google Drive mounted successfully")

##### Download the dataset to Colab's local storage. This is faster than reading it from Drive and avoids filling Drive with temporary files.

In [ ]:
import os

# Create a directory for the dataset
os.makedirs('/content/datasets/VisDrone', exist_ok=True)

!wget -q -O /content/datasets/VisDrone/train.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!unzip -q -o /content/datasets/VisDrone/train.zip -d /content/datasets/VisDrone/
print("Train verisi indiriliyor")

!wget -q -O /content/datasets/VisDrone/val.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!unzip -q -o /content/datasets/VisDrone/val.zip -d /content/datasets/VisDrone/
print("Validate verisi indiriliyor")

# Remove the ZIP files to save disk space
!rm /content/datasets/VisDrone/*.zip

print("Veri seti Colab'e indirildi")

---
# **SECTION 2 — Dataset Preparation**
---

##### The original dataset has 10 classes, but this project focuses on people. We keep the pedestrian and people classes, merge them into one `Human` class, skip ignored annotations with score 0, and convert the remaining labels to YOLO format.

In [ ]:
import os
from PIL import Image

def visdrone_to_yolo_human_only(base_dir):
    images_dir = os.path.join(base_dir, 'images')
    annotations_dir = os.path.join(base_dir, 'annotations')
    labels_dir = os.path.join(base_dir, 'labels')

    if not os.path.exists(annotations_dir): return
    os.makedirs(labels_dir, exist_ok=True)
    print(f"Processing: {base_dir}")

    file_count = 0
    for img_name in os.listdir(images_dir):
        if not img_name.endswith('.jpg'): continue

        ann_path = os.path.join(annotations_dir, img_name.replace('.jpg', '.txt'))
        label_path = os.path.join(labels_dir, img_name.replace('.jpg', '.txt'))

        if not os.path.exists(ann_path): continue

        with Image.open(os.path.join(images_dir, img_name)) as img:
            img_width, img_height = img.size

        new_lines = []
        with open(ann_path, 'r', encoding='utf-8') as f:
            for line in f.readlines():
                parts = line.strip().split(',')
                if len(parts) < 8: continue

                # Skip ignored regions (score = 0)
                score = int(parts[4])
                if score == 0: continue

                # Merge class 1 (pedestrian) and class 2 (people)
                category = int(parts[5])
                if category in [1, 2]:
                    x_center = (float(parts[0]) + float(parts[2]) / 2.0) / img_width
                    y_center = (float(parts[1]) + float(parts[3]) / 2.0) / img_height
                    w = float(parts[2]) / img_width
                    h = float(parts[3]) / img_height
                    # Save both source classes as class 0
                    new_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        if len(new_lines) > 0:
            with open(label_path, 'w', encoding='utf-8') as f:
                f.writelines(new_lines)
            file_count += 1

    print(f"{file_count} images converted to YOLO format.")

# Run the conversion function
visdrone_to_yolo_human_only('/content/datasets/VisDrone/VisDrone2019-DET-train')
visdrone_to_yolo_human_only('/content/datasets/VisDrone/VisDrone2019-DET-val')

##### Create a YAML configuration that tells the model where the training data is and which class to detect. Save the file to Drive so it persists.

In [ ]:
import os

# Create the dataset YAML
yaml_content = """
path: /content/datasets/VisDrone
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
nc: 1
names:
  0: Human
"""

# Save the YAML file to Google Drive
hedef_klasor = '/content/drive/MyDrive/Tez_YOLO11_Projesi'
os.makedirs(hedef_klasor, exist_ok=True)

yaml_yolu = os.path.join(hedef_klasor, 'custom_visdrone.yaml')
with open(yaml_yolu, 'w') as f:
    f.write(yaml_content.strip())
print("custom_visdrone.yaml is ready")

##### YOLO11m normally uses P3, P4, and P5 detection levels, corresponding to strides of 8, 16, and 32 pixels. People can occupy very small regions in drone images, so these levels may miss fine details. This project adds a P2 level with a stride of 4 to retain higher-resolution features for small objects.

In [ ]:
yaml_content = """# Ultralytics YOLO, AGPL-3.0 license
# YOLO11 object detection model with P2-P5 outputs.

nc: 1
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]          # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]          # 16 (P3/8)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]          # 19 (P2/4)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]          # 22 (P3/8)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]          # 25 (P4/16)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]          # 28 (P5/32)

  - [[19, 22, 25, 28], 1, Detect, [nc]]
"""

with open('yolo11m-p2.yaml', 'w') as f:
    f.write(yaml_content)

print("yolo11m-p2.yaml created")



---
# **SECTION 3 — Training**
---

##### Train the model. YOLO11m is used as the starting model, and a P2 detection level is added because the standard detection levels may not capture small people well enough in drone imagery.

In [ ]:
BATCH      = 24 # The GPU processes 24 images per batch
WORKERS    = 12 # Number of background data-loading workers
CACHE      = 'ram' # Cache the dataset in RAM for faster loading
YAML_PATH  = '/content/drive/MyDrive/Tez_YOLO11_Projesi/custom_visdrone.yaml' #YAML file that describes the dataset to the model
OUTPUT_DIR = '/content/drive/MyDrive/Tez_YOLO11_Projesi' #Directory where training outputs are saved

import os

from ultralytics import YOLO
# Use the P2 detection level here
model = YOLO('yolo11m-p2.yaml').load('yolo11m.pt')
print("Model loaded")
model.info()

print(f"\n Training starting batch={BATCH} | imgsz=832 | cache='{CACHE}'")
print("─" * 60)


# Training parameters

results = model.train(
    data     = YAML_PATH,   # YAML file
    epochs   = 150,         # Target 150 training epochs
    imgsz    = 832,         # Resize images to 832 × 832
    batch    = BATCH,       # Batch size is 24 for the A100; GPU memory issues occurred three times during training
    device   = 0,           # Use the GPU (A100) for training
    workers  = WORKERS,     # Number of data-loading workers
    project  = OUTPUT_DIR,  # Output directory
    name     = 'YOLO11m_P2_832_Final', # Directory name for all training outputs
    exist_ok = True, # Reuse the directory if it already exists

    cache    = CACHE,

    # Image augmentation settings
    hsv_h        = 0.015,   # Slightly vary hue to simulate camera and lens differences
    hsv_s        = 0.7,     # Vary color saturation
    hsv_v        = 0.4,     # Vary brightness
    degrees      = 0.0,     # Disable rotation; the images are captured from above
    translate    = 0.1,     # Slight translation to simulate drone movement
    scale        = 0.2,     # Preserve small-object detail
    fliplr       = 0.5,     # Apply horizontal flips
    flipud       = 0.0,     # Disable vertical flips; they are not useful for these drone images
    mosaic       = 1.0,     # Combine four drone images into a mosaic during training
    copy_paste   = 0.2,     # Copy and paste people into other image regions
    mixup        = 0.05,    # Blend two images to create more complex scenes
    erasing      = 0.0,     # Disable random erasing, which could remove small objects
    close_mosaic = 10,      # Disable mosaic augmentation for the final 10 epochs

    # Optimization settings
    optimizer     = 'AdamW',  # Optimizer choice
    lr0           = 0.003,    # Initial learning rate
    lrf           = 0.01,     # Multiplier applied to the initial learning rate at the end of training
    warmup_epochs = 3,        # Use 3 warm-up epochs
    weight_decay  = 0.0005,   # Weight decay to help regularize training

    # Loss weights
    box = 7.5,  # Bounding-box regression loss weight
    cls = 0.5,  # Classification loss weight (single-class task)
    dfl = 1.5,  # Distribution focal loss weight for box localization

    # Mixed-precision training
    amp = True, # Enable automatic mixed precision (FP16 where supported)

    # Training control settings
    patience    = 40, # Stop if validation does not improve for 40 epochs
    save_period = 10, # Save a checkpoint every 10 epochs
    plots       = True, # Generate result plots
    verbose     = True, # Print training progress and operations
)

print("\n" + "═" * 60)
print("TRAINING COMPLETE")
best = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')
if os.path.exists(best):
    print(f"best.pt → {best} ({os.path.getsize(best)/1e6:.1f} MB)")
print("═" * 60)

---
# **SECTION 4 — Validation**
---

##### Validate the trained model on the validation split.

In [ ]:
!pip install ultralytics -q

In [ ]:
BATCH      = 24
OUTPUT_DIR = '/content/drive/MyDrive/Tez_YOLO11_Projesi'
YAML_PATH  = '/content/drive/MyDrive/Tez_YOLO11_Projesi/custom_visdrone.yaml'

import os,json
from ultralytics import YOLO

# Load the best checkpoint produced during training
BEST_PT = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')

if not os.path.exists(BEST_PT):
    print(f"best.pt not found: {BEST_PT}. Complete SECTION 3 (training) first.")
else:
    model = YOLO(BEST_PT)
    metrics = model.val(
        data    = YAML_PATH,
        imgsz   = 832,
        batch   = BATCH,
        device  = 0,
        split   = 'val',  # Evaluate on the validation split
        plots   = True,
        verbose = True,
    )
    print("\n" + "═" * 50)
    print("RESULTS")
    print("═" * 50)
    print(f"  mAP@0.50      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.50:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision     : {metrics.box.mp:.4f}")
    print(f"  Recall        : {metrics.box.mr:.4f}")

    results_dict = {
        "mAP50":      round(metrics.box.map50, 4),
        "mAP50-95":   round(metrics.box.map, 4),
        "Precision":  round(metrics.box.mp, 4),
        "Recall":     round(metrics.box.mr, 4),
    }

    save_path = os.path.join(OUTPUT_DIR, 'val_results.json')
    with open(save_path, 'w') as f:
        json.dump(results_dict, f, indent=2)

---
# **SECTION 5 — Video Preprocessing**
---

####**This section is optional, but recommended.**
##### Preprocess test videos into a format OpenCV can read. Some video encodings can cause issues, so this cell converts the input video before inference.

In [ ]:
# Upload a test video named test_video.mp4 using the Colab file panel
import os, subprocess

VIDEO_INPUT = '/content/test_video.mp4'    # Input video path
VIDEO_FIXED = '/content/test_video_fixed.mp4' # Path for the converted video

if not os.path.exists(VIDEO_INPUT):
    print(f"Video not found: {VIDEO_INPUT}")
else:
    print(f"Kaynak: {VIDEO_INPUT}")

    result = subprocess.run([
        'ffmpeg', '-y', '-i', VIDEO_INPUT,
        '-c:v', 'libx264', '-crf', '23', '-preset', 'fast', # Convert to H.264 while preserving visual quality
        '-c:a', 'aac', '-b:a', '128k',  # Convert audio to the standard AAC format
        '-movflags', '+faststart',  # Move the video index to the beginning for faster playback startup
        VIDEO_FIXED
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f"Conversion complete: {VIDEO_FIXED}")
    else:
        print(f"FFmpeg error:\n{result.stderr[-2000:]}")

---
# **SECTION 6 — ByteTrack Inference**
---

##### Define custom settings for the tracking algorithm instead of relying on its defaults.

In [ ]:
import yaml

custom_tracker = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.4,   # Threshold for matching an existing track
    "track_low_thresh": 0.1,    # Keep the track while confidence remains above 10%
    "new_track_thresh": 0.5,    # Require at least 50% confidence to create a new track
    "track_buffer": 90,         # Keep lost tracks for 90 frames (3 seconds at 30 FPS)
    "match_thresh": 0.8         # Maximum matching distance between boxes in adjacent frames
}

with open('custom_bytetrack.yaml', 'w') as f:
    yaml.dump(custom_tracker, f)

print("custom_bytetrack.yaml created")

##### Run the detector and assign persistent tracking IDs to people. These settings are a prototype and may need adjustment for real-world use; see the notes in the code.

In [ ]:
# This prototype was created to be demonstrated in the current environment
# Porting the implementation to C++ could improve performance
# For embedded deployment, export the model to TensorRT (INT8); this project uses FP16

import cv2, os, random
from ultralytics import YOLO
from datetime import datetime

# File paths
MODEL_PATH = os.path.join(OUTPUT_DIR, 'YOLO11m_P2_832_Final', 'weights', 'best.pt')
VIDEO_IN   = '/content/test_video_fixed.mp4'
OUTPUT_VID = os.path.join(OUTPUT_DIR, 'Video_Sonuclari', 'Tez_ByteTrack_Sonuc.mp4')
os.makedirs(os.path.dirname(OUTPUT_VID), exist_ok=True)

for label, path in [('Model', MODEL_PATH), ('Video', VIDEO_IN)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f" {label} not found: {path}")

# Prepare the video and model
model = YOLO(MODEL_PATH)
cap   = cv2.VideoCapture(VIDEO_IN)
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS   = int(cap.get(cv2.CAP_PROP_FPS)) or 30
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {W}×{H} @ {FPS} FPS — {TOTAL} kare")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VID, fourcc, FPS, (W, H))
if not out.isOpened():
    raise RuntimeError("Could not open VideoWriter")

random.seed(42)
PALETTE = {i: (random.randint(50,255), random.randint(50,255), random.randint(50,255))
           for i in range(1000)}

print("\nStarting ByteTrack")
print("─" * 60)

t0      = datetime.now()
n_frame = 0
id_set  = set()

# Frame-by-frame tracking loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Detect objects and assign tracking IDs
    res = model.track(
        frame,
        conf    = 0.20, # Use a lower confidence threshold to retain partially occluded people
        iou     = 0.30, # Suppress duplicate boxes when their overlap exceeds the IoU threshold
        tracker = 'custom_bytetrack.yaml', # Use the custom tracker settings
        persist = True, # Persist track IDs across frames
        verbose = False,
        imgsz   = 832,
        device  = 0, # Use the GPU
        augment=True, # Test-time augmentation; disable for better real-time speed
    )

    if res[0].boxes.id is not None:
        boxes  = res[0].boxes.xyxy.cpu().numpy()
        ids    = res[0].boxes.id.int().cpu().tolist()
        scores = res[0].boxes.conf.cpu().tolist()
        for box, tid, sc in zip(boxes, ids, scores):
            x1, y1, x2, y2 = map(int, box)
            cx, cy = (x1+x2)//2, (y1+y2)//2
            color  = PALETTE[tid % 1000]
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 1)
            cv2.circle(frame, (cx,cy), 3, color, -1)  # Object center point
            cv2.putText(frame, f"ID:{tid} {sc:.2f}", (x1, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1, cv2.LINE_AA)
            id_set.add(tid)

    out.write(frame)
    n_frame += 1
    if n_frame % 100 == 0:
        elapsed = max((datetime.now()-t0).seconds, 1)
        fps_avg = n_frame / elapsed
        remain  = (TOTAL - n_frame) / fps_avg
        print(f"  Frame {n_frame}/{TOTAL} | {fps_avg:.1f} FPS | ~{remain:.0f}s remaining")

cap.release()
out.release()

sure = (datetime.now()-t0).total_seconds()
print("\n" + "═"*60)
print("INFERENCE COMPLETE")
print(f"  Frames processed  : {n_frame}")
print(f"  Benzersiz ID  : {len(id_set)}")
print(f"  Total time   : {sure:.1f} s  |  Avg. {n_frame/sure:.1f} FPS")
print(f"  Output         : {OUTPUT_VID}")
print("═"*60)

## Citation

VisDrone-DET2019: Zhu, Pengfei, et al. “Detection and Tracking Meet Drones Challenge.” *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 44(11), 7380–7399 (2022). [DOI: 10.1109/TPAMI.2021.3119563](https://doi.org/10.1109/TPAMI.2021.3119563).

Citation does not replace permission to use or redistribute the dataset.